# Cross-Trajectory Warm Start Comparison

Take the last timestep of iter 13 and iter 40, run both forward under iter 13 conditions.
Compare convergence.


In [ ]:
%load_ext autoreload
%autoreload 2

import sys, os
sys.path.append("..")
os.environ["CUDA_VISIBLE_DEVICES"] = "7" 


In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

from neurips_diff_eval import (
    to_model_space,
    from_model_space,
    run_trajectory_pair,
    time_to_convergence,
    load_reference_flux,
)


## Config


In [ ]:
GKW_RAW_DIR = "/restricteddata/ukaea/gyrokinetics/raw"

TRAJ_TGT = 13   # target conditions & ground truth
TRAJ_SRC = 40   # source of the "foreign" warm start IC

N_EVAL_STEPS = 5000
CHUNK_SIZE = 50
BACKEND = "jax"
K_INDEX = 80


## 1. Load raw GKW data


In [ ]:
from gyaradax import gk_from_gkw_dir

df_tgt, geom_tgt, params_tgt, state_tgt, pre_tgt = gk_from_gkw_dir(
    os.path.join(GKW_RAW_DIR, f"iteration_{TRAJ_TGT}"),
    mixed_precision=True, k_index=K_INDEX,
)
print(f"Target (iter {TRAJ_TGT}): rlt={params_tgt.rlt:.4f}, rln={params_tgt.rln:.4f}, "
      f"shat={params_tgt.shat:.4f}, q={params_tgt.q:.4f}")

df_src, geom_src, params_src, state_src, pre_src = gk_from_gkw_dir(
    os.path.join(GKW_RAW_DIR, f"iteration_{TRAJ_SRC}"),
    mixed_precision=True, k_index=K_INDEX,
)
print(f"Source (iter {TRAJ_SRC}): rlt={params_src.rlt:.4f}, rln={params_src.rln:.4f}, "
      f"shat={params_src.shat:.4f}, q={params_src.q:.4f}")

ic_tgt = df_tgt
ic_src = df_src
rel_diff = np.linalg.norm(np.array(ic_tgt) - np.array(ic_src)) / np.linalg.norm(np.array(ic_tgt))
print(f"Rel L2 between ICs: {rel_diff:.4f}")


## 2. Run warm starts


In [ ]:
log_gt, log_self = run_trajectory_pair(
    df_tgt, ic_tgt, geom_tgt, params_tgt, pre_tgt, state_tgt,
    N_EVAL_STEPS, f"self-{TRAJ_TGT}", chunk_size=CHUNK_SIZE,
    backend=BACKEND, print_every=500,
)

_, log_cross = run_trajectory_pair(
    df_tgt, ic_src, geom_tgt, params_tgt, pre_tgt, state_tgt,
    N_EVAL_STEPS, f"cross-{TRAJ_SRC}->{TRAJ_TGT}", chunk_size=CHUNK_SIZE,
    backend=BACKEND, print_every=500,
)

print("Both simulations complete.")


## 3. Time to convergence


In [ ]:
ref_mean, ref_std = load_reference_flux(GKW_RAW_DIR, TRAJ_TGT)
t = log_gt["time"]

ttc_self = time_to_convergence(log_self, log_gt, t, ref_flux_mean=ref_mean, ref_flux_std=ref_std)
ttc_cross = time_to_convergence(log_cross, log_gt, t, ref_flux_mean=ref_mean, ref_flux_std=ref_std)

print(f"\nSelf  (iter {TRAJ_TGT}): TTC_flux={ttc_self['flux']:.1f}, TTC_spec={ttc_self['ky_spec']:.1f}")
print(f"Cross (iter {TRAJ_SRC}): TTC_flux={ttc_cross['flux']:.1f}, TTC_spec={ttc_cross['ky_spec']:.1f}")


## 4. Comparison plots


In [ ]:
snap_colors = ["#2a9d8f", "#e76f51", "#264653"]

fig, axes = plt.subplots(2, 3, figsize=(18, 9), sharex="col")
labels = [f"self (iter {TRAJ_TGT})", f"cross (iter {TRAJ_SRC})"]
all_logs = [log_self, log_cross]
ttcs = [ttc_self, ttc_cross]

for row, (lw, ttc, label) in enumerate(zip(all_logs, ttcs, labels)):
    lg = log_gt
    t = lg["time"]
    n_t = len(t)

    snap_idx = [min(5, n_t - 1), n_t // 2, n_t - 1]
    for j, si in enumerate(snap_idx):
        ky_gt = np.log10(np.maximum(lg["ky_spec"][si], 1e-30))
        ky_w = np.log10(np.maximum(lw["ky_spec"][si], 1e-30))
        lbl = f"t={float(t[si]):.1f}"
        axes[row, 0].plot(ky_gt, "-", color=snap_colors[j], lw=1.2, alpha=0.7, label=f"GT {lbl}")
        axes[row, 0].plot(ky_w, "--", color=snap_colors[j], lw=1.2, alpha=0.9, label=f"warm {lbl}")
    axes[row, 0].set(title=f"{label}: $k_y$ spectrum", ylabel="log$_{10}$(E)")
    axes[row, 0].legend(fontsize=6, ncol=2); axes[row, 0].grid(True, alpha=0.15)

    axes[row, 1].plot(t, lg["eflux"], "k", lw=0.8, alpha=0.5, label="GT")
    axes[row, 1].plot(t, lw["eflux"], lw=1, label="warm")
    axes[row, 1].axhspan(ref_mean - ref_std, ref_mean + ref_std, color="k", alpha=0.08, label="ref ±1σ")
    axes[row, 1].axhline(ref_mean, color="k", ls=":", lw=0.8)
    if ttc["flux"] < np.inf:
        axes[row, 1].axvline(t[0] + ttc["flux"], color="green", ls="-", lw=1.5,
                              label=f"TTC_flux={ttc['flux']:.1f}")
    if ttc["ky_spec"] < np.inf:
        axes[row, 1].axvline(t[0] + ttc["ky_spec"], color="blue", ls="--", lw=1.5,
                              label=f"TTC_spec={ttc['ky_spec']:.1f}")
    axes[row, 1].set_title(f"{label}: flux"); axes[row, 1].legend(fontsize=6)
    axes[row, 1].grid(True, alpha=0.15)

    n_compare = min(len(lw["ky_spec"]), len(lg["ky_spec"]))
    rel_l2 = []
    for i in range(n_compare):
        ky_w = np.log10(np.maximum(lw["ky_spec"][i], 1e-30))
        ky_g = np.log10(np.maximum(lg["ky_spec"][i], 1e-30))
        norm_g = np.linalg.norm(ky_g)
        rel_l2.append(np.linalg.norm(ky_w - ky_g) / norm_g if norm_g > 0 else 1.0)
    axes[row, 2].plot(t[:n_compare], rel_l2, lw=1, color="#264653")
    axes[row, 2].axhline(0.05, color="green", ls="--", lw=0.8, label="5%")
    axes[row, 2].axhline(0.10, color="orange", ls="--", lw=0.8, label="10%")
    if ttc["ky_spec"] < np.inf:
        axes[row, 2].axvline(t[0] + ttc["ky_spec"], color="blue", ls="--", lw=1, alpha=0.5)
    axes[row, 2].set_title(f"{label}: rel L2($k_y$)")
    axes[row, 2].set_ylim(0, max(0.5, np.max(rel_l2) * 1.1))
    axes[row, 2].legend(fontsize=7); axes[row, 2].grid(True, alpha=0.15)

for ax in axes[-1]:
    ax.set_xlabel(r"time $[v_{th}/R]$")

fig.suptitle(f"Warm start: self (iter {TRAJ_TGT}) vs cross (iter {TRAJ_SRC} -> {TRAJ_TGT})",
             fontsize=13, fontweight="bold", y=1.01)
fig.tight_layout()


## 5. Overlay flux


In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
t = log_gt["time"]
ax.plot(t, log_gt["eflux"], "k", lw=0.8, alpha=0.4, label="GT")
ax.plot(t, log_self["eflux"], lw=1.2, label=f"self (iter {TRAJ_TGT})", color="#2a9d8f")
ax.plot(t, log_cross["eflux"], lw=1.2, label=f"cross (iter {TRAJ_SRC})", color="#e76f51")
ax.axhspan(ref_mean - ref_std, ref_mean + ref_std, color="k", alpha=0.06)
ax.axhline(ref_mean, color="k", ls=":", lw=0.8)

for ttc, color, ls in [(ttc_self, "#2a9d8f", "-"), (ttc_cross, "#e76f51", "--")]:
    if ttc["flux"] < np.inf:
        ax.axvline(t[0] + ttc["flux"], color=color, ls=ls, lw=1.5, alpha=0.7)

ax.set(xlabel=r"time $[v_{th}/R]$", ylabel="eflux",
       title=f"Flux: self vs cross warm start (target=iter {TRAJ_TGT})")
ax.legend(fontsize=9); ax.grid(True, alpha=0.15)
fig.tight_layout()


## 6. Summary


In [ ]:
print(f"{'Metric':<20} {'Self':>12} {'Cross':>12}")
print("-" * 46)
print(f"{'TTC flux':<20} {ttc_self['flux']:>12.1f} {ttc_cross['flux']:>12.1f}")
print(f"{'TTC spec':<20} {ttc_self['ky_spec']:>12.1f} {ttc_cross['ky_spec']:>12.1f}")

n_tail = 80
self_final = np.mean(log_self["eflux"][-n_tail:])
cross_final = np.mean(log_cross["eflux"][-n_tail:])
print(f"{'Final flux (mean)':<20} {self_final:>12.3e} {cross_final:>12.3e}")
print(f"{'Ref flux':<20} {ref_mean:>12.3e} {'':>12}")
print(f"{'IC rel L2 diff':<20} {0.0:>12.4f} {rel_diff:>12.4f}")
